In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

In [ ]:
import celldega as dega
from pathlib import Path
import pandas as pd
import matplotlib
import os
import numpy as np
import scanpy as sc
from ipywidgets import Widget
import alphashape
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
import geopandas as gpd
print(dega.__version__)

In [ ]:
Widget.close_all()

## Inputs

In [ ]:
# technology = "Xenium"
# # sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
# sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
# data_dir = f"data/xenium_data/{sample}"
# segmentation_suffix = ""
# # path_landscape_files = f'data/landscape_files/{sample}_09-29-25'
# path_landscape_files = f'data/landscape_files/{sample}_09-29-25'
# base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

In [ ]:
technology = "Xenium"
# sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
sample = 'Xenium_V1_human_Pancreas_FFPE_outs'
data_dir = f"data/xenium_data/{sample}"
segmentation_suffix = ""
# path_landscape_files = f'data/landscape_files/{sample}_09-29-25'
path_landscape_files = f'data/landscape_files/{sample}'
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

## Make AnnData

In [ ]:
f'{path_landscape_files}/{Path(path_landscape_files).name}.h5ad'

In [ ]:
# Load h5ad file, if exists already
adata = sc.read_h5ad("data/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs.h5ad")

In [ ]:
# # Load h5ad file, if exists already
# adata = sc.read_h5ad(f'{path_landscape_files}/{Path(path_landscape_files).name}.h5ad')

In [ ]:
# import argparse
# from collections import defaultdict
# from pathlib import Path
# import shutil

# import pandas as pd
# import spatialdata as sd
# from spatialdata_io import xenium

# import celldega as dega
# def _make_xenium_anndata(data_dir, path_landscape_files, write=True):
#     path_landscape_files = Path(path_landscape_files)
#     sample = path_landscape_files.name

#     zarr_path = path_landscape_files / f"{sample}.zarr"

#     # check if the zarr file already exists
#     if zarr_path.exists():
#         print(f"The file {zarr_path} already exists.")
#         sdata = sd.read_zarr(zarr_path)
#     else:
#         # ingest xenium data raw output folder using spatialdata-io
#         sdata = xenium(data_dir)
#         sdata.write(zarr_path)
#         print(f"Data written to {zarr_path} successfully.")

#     # create anndata from sdata.tables["table"]
#     adata = sdata.tables["table"]

#     if write:
#         h5ad_path = path_landscape_files / f"{sample}.h5ad"
#         adata.write_h5ad(h5ad_path)
#         print(f"AnnData written to {h5ad_path}")

#     return adata

In [ ]:
# adata = _make_xenium_anndata(data_dir, path_landscape_files, write=True)

## Add default leiden clustering to the Anndata

In [ ]:
meta_cell = pd.read_parquet("data/landscape_files/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs/cell_metadata.parquet")
default_clustering, _clusters, _ser_counts = dega.pre._load_xenium_cluster_data("../../../Downloads/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs",
meta_cell)

cluster_to_cell_id = default_clustering["cluster"].to_dict()
adata.obs["leiden"] = adata.obs["cell_id"].map(cluster_to_cell_id)

In [ ]:
# meta_cell = pd.read_parquet(f"{path_landscape_files}/cell_metadata{segmentation_suffix}.parquet")
# default_clustering, _clusters, _ser_counts = dega.pre._load_xenium_cluster_data(data_dir, meta_cell)

# cluster_to_cell_id = default_clustering["cluster"].to_dict()
# adata.obs["leiden"] = adata.obs["cell_id"].map(cluster_to_cell_id)

## Rank and save marker genes

In [ ]:
n_genes = 100

In [ ]:
# # Run ranking
# sc.tl.rank_genes_groups(adata, groupby="leiden", method="t-test", use_raw=False, show_progress=True)

# # Save markers
# marker_df = dega.qc._get_ranked_genes_df(adata, n_genes)
# marker_df.to_csv(f"{path_landscape_files}/marker_genes_by_cluster-{n_genes}_genes.csv", index=False)

### Next Steps:

#### 1. Uploaded "marker_genes_by_cluster-{n_genes}_genes.csv" on ChatGPT, and asked for tentative cell types.
#### 2. "Predicted_Cell_Types_Def_Clustering-{n_genes}_genes.csv" has the predicted cell types for each cluster based on the top 10 marker genes, with the cluster number included in the label.

In [ ]:
# cell types

pred_cell_types_df = pd.read_csv(f"{path_landscape_files}/Predicted_Cell_Types_Def_Clustering-{n_genes}_genes.csv")

pred_cell_types_df.drop(['Unnamed: 0'], axis=1, inplace=True)
pred_cell_types_df["category"] = pred_cell_types_df["predicted_cell_type"].str.split("_").str[0]
pred_cell_types_df['cluster'] = pred_cell_types_df['cluster'].astype('string')
pred_cell_types_df.set_index('cluster', inplace=True)

cluster_to_celltype = pred_cell_types_df["predicted_cell_type"].to_dict()

pred_cell_types_df.reset_index(inplace=True)
pred_cell_types_df.head()

## Read cluster related files generated by Celldega 

In [ ]:
# cluster = pd.read_parquet(f"{path_landscape_files}/cell_clusters/cluster.parquet")
# meta_cluster = pd.read_parquet(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet")
# df_sig = pd.read_parquet(f"{path_landscape_files}/df_sig.parquet")

In [ ]:
path_landscape_files = "data/landscape_files/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs"
cluster = pd.read_parquet(f"{path_landscape_files}/cell_clusters/cluster.parquet")
meta_cluster = pd.read_parquet(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet")
df_sig = pd.read_parquet(f"{path_landscape_files}/df_sig.parquet")

## Modify cluster related files generated by Celldega, to include cell types

In [ ]:
# cluster["predicted_cell_type"] = cluster["cluster"].map(cluster_to_celltype)
# cluster.fillna('None')
# cluster.drop(['cluster'], axis=1, inplace=True)
# cluster.rename(columns={'predicted_cell_type':'cluster'}, inplace=True)

# os.rename(f"{path_landscape_files}/cell_clusters/cluster.parquet",
#           f"{path_landscape_files}/cell_clusters/cluster_default.parquet")

# cluster.to_parquet(f"{path_landscape_files}/cell_clusters/cluster.parquet")

In [ ]:
# meta_cluster.index = meta_cluster.index.astype('string')
# meta_cluster.index = meta_cluster.index.map(cluster_to_celltype)

# os.rename(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet",
#           f"{path_landscape_files}/cell_clusters/meta_cluster_default.parquet")

# meta_cluster.to_parquet(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet")

In [ ]:
# df_sig.columns = df_sig.columns.astype('string')
# df_sig.columns = df_sig.columns.map(cluster_to_celltype)

# os.rename(f"{path_landscape_files}/df_sig.parquet",
#           f"{path_landscape_files}/df_sig_default.parquet")

# df_sig.to_parquet(f"{path_landscape_files}/df_sig.parquet")

### Make hextiles

In [ ]:
adata.obs['leiden'] = adata.obs['cell_id'].map(cluster['cluster'])

In [ ]:
data = dega.nbhd._get_gdf_cell(adata)
gdf_nbhd = dega.nbhd.generate_hex_grid(data, diameter=75)

## Trx-assignment-proportion per Hextile using NBHD module methods

In [ ]:
gdf_trx = dega.nbhd._get_gdf_trx(data_dir)

In [ ]:
# nbhd_meta = dega.nbhd.get_nbhd_meta(gdf_nbhd = gdf_nbhd,
#                                     unique_nbhd_col="name",
#                                     gdf_trx = gdf_trx,
#                                     gdf_cell = data)

# nbhd_meta.head()

In [ ]:
# nbhd_meta_TEST = nbhd_meta[nbhd_meta['unassigned_trx_pct'] < 0.75]

In [ ]:
# nbhd_meta['unassigned_trx_prop'] = np.where(
#     nbhd_meta['total_trx'] > 0,
#     nbhd_meta['unassigned_trx_count'] / nbhd_meta['total_trx'],
#     np.nan
# )

# nbhd_meta['unassigned_trx_prop'].fillna(0, inplace=True)
# nbhd_meta.head()

In [ ]:
# gdf_nbhd['assigned_trx_pct'] = gdf_nbhd['name'].map(nbhd_meta['assigned_trx_pct'])
# gdf_nbhd['total_trx'] = gdf_nbhd['name'].map(nbhd_meta['total_trx'])
# gdf_nbhd.head()

## Cell-cluster by Hextile using NBHD module methods

In [ ]:
adata_nbp, gdf_nbhd = dega.nbhd.calc_nbp(data, gdf_nbhd, category="cluster")

In [ ]:
# Clustering
sc.pp.normalize_total(adata_nbp, inplace=True)
sc.pp.log1p(adata_nbp)
sc.pp.neighbors(adata_nbp, n_neighbors=10)
sc.tl.leiden(adata_nbp, resolution=1)

In [ ]:
population_distribution = pd.DataFrame(
    adata_nbp.X, index=adata_nbp.obs_names, columns=adata_nbp.var_names
)

In [ ]:
# Add clustering and proportions to hex GeoDataFrame
gdf_nbhd = gdf_nbhd.set_index("name")
gdf_nbhd["leiden"] = gdf_nbhd.index.map(adata_nbp.obs['leiden'])
gdf_nbhd["niche"] = [f"{cluster}" for cluster in gdf_nbhd["leiden"].values]
gdf_nbhd = gdf_nbhd.join(population_distribution)
gdf_nbhd.reset_index(inplace=True)

In [ ]:
# Dissolve to form niches
gdf_niche = dega.nbhd._dissolve_by_category(gdf_nbhd, "leiden")
gdf_niche["name"] = [f"{c}" for c in gdf_niche["leiden"]]
gdf_niche.head()

## Clustergram: cell_population-by-hextile_nbhd 

In [ ]:
gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
gdf_nbhd_.set_index('name', inplace=True)
gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
gdf_nbhd_.head()

In [ ]:
gdf_nbhd_T = gdf_nbhd_.T
gdf_nbhd_T.head()

In [ ]:
meta_col = pd.DataFrame(index=gdf_nbhd_T.columns.tolist())
top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index.tolist()
niches = gdf_nbhd.set_index("name").loc[top_cols, "niche"]
niches = pd.DataFrame(niches)
meta_col["niche"] = meta_col.index.map(niches["niche"])
meta_col[:5]

In [ ]:
meta_row = pd.DataFrame(index=gdf_nbhd_T.index.tolist())
top_rows = [row for row in gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index]
meta_row["category"] = meta_row.index

In [ ]:
mat = dega.clust.Matrix(
    gdf_nbhd_T,
    name='parquet',
    meta_col=meta_col,
    col_attr=['niche'],
    meta_row=meta_row,
    row_entity="cell_cluster",
    col_entity="nbhd"
)

mat.downsample_to(axis='col', category='niche')
mat.norm(axis='row', by='zscore')
mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat,
    width=500,
    height=500,
)

## Visualize in Landscape view: Hextile NBHD

### Trx-assignment per Hextile

In [ ]:
# geometry_series = data['geometry'] # gdf_cell
# valid_geoms = geometry_series.dropna()
# points = np.array([[point.centroid.x, point.centroid.y] for point in valid_geoms])

In [ ]:
# ashape = alphashape.alphashape(points, 0.02)
# ashape

In [ ]:
# largest = max(ashape.geoms, key=lambda p: p.area)
# largest_gdf = gpd.GeoDataFrame(
#     {"geometry": [largest]}
# )

In [ ]:
# gdf_nbhd_LF = gdf_nbhd.copy()
# gdf_nbhd_LF.shape

In [ ]:
gdf_test = gdf_trx.sjoin(gdf_nbhd, how="inner", predicate="within")
gdf_test.head()

In [ ]:
counts = gdf_test.groupby("name")['cell_id'].agg(
        total="size", unassigned=lambda x: (x == "UNASSIGNED").sum()
    )

In [ ]:
percentage_unassigned = (counts["unassigned"] / counts["total"]) * 100

percentage_unassigned = percentage_unassigned.fillna(0)

percentage_unassigned = percentage_unassigned[
    percentage_unassigned < 75
]

In [ ]:
df_test=pd.DataFrame(percentage_unassigned)
df_test.rename(columns={0:'percentage_unassigned'}, inplace=True)
df_test.head()

In [ ]:
# gdf_nbhd = dega.nbhd.generate_hex_grid(data, diameter=75)
gdf_nbhd.set_index('name', inplace=True)

In [ ]:
gdf_nbhd

In [ ]:
df_test['geometry'] = gdf_nbhd.loc[df_test.index, 'geometry'].values
df_test = gpd.GeoDataFrame(df_test, geometry='geometry', crs=gdf_nbhd.crs)
df_test.head()

In [ ]:
df_test.shape

In [ ]:
# Z-score standardization
values = df_test['percentage_unassigned']
zscore = (values - values.mean()) / values.std()

# Optionally, rescale z-scores to 0–1 for color mapping
# This keeps color mapping compatible with the colormap range [0, 1]
# zscore_scaled = (zscore - zscore.min()) / (zscore.max() - zscore.min())

zscore_clipped = np.clip(zscore, -1, 1)  # focus on ±2 SDs
zscore_scaled = (zscore_clipped - zscore_clipped.min()) / (zscore_clipped.max() - zscore_clipped.min())

# Define a custom red-gray-blue colormap
colors = [
    (0.0, "#d62728"),  # red (high assignment)
    (0.5, "#7f7f7f"),  # gray (mid)
    (1.0, "#1f77b4"),  # blue (low assignment) 
]
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("red_gray_blue", colors)

df_test['color'] = [matplotlib.colors.to_hex(cmap(v)) for v in zscore_scaled]

df_test['cat'] = np.select(
    [zscore_scaled > 0.66, zscore_scaled > 0.33],
    ['Low-Assignment', 'Mid-Assignment'],
    default='High-Assignment'
)

df_test['area'] = df_test['geometry'].area

df_test.head()

In [ ]:
df_test.to_parquet(f"{path_landscape_files}/gdf_nbhd_meta.parquet")

In [ ]:
# nbhd_meta_TEST['geometry'] = nbhd_meta_TEST.index.map(gdf_nbhd['geometry'])
# nbhd_meta_TEST = gpd.GeoDataFrame(nbhd_meta_TEST, geometry='geometry', crs=gdf_nbhd.crs)
# nbhd_meta_TEST.shape

In [ ]:
# nbhd_meta_TEST.columns

In [ ]:
# # Z-score standardization
# values = nbhd_meta_TEST['unassigned_trx_pct']
# zscore = (values - values.mean()) / values.std()

# # Optionally, rescale z-scores to 0–1 for color mapping
# # This keeps color mapping compatible with the colormap range [0, 1]
# # zscore_scaled = (zscore - zscore.min()) / (zscore.max() - zscore.min())

# zscore_clipped = np.clip(zscore, -1, 1)  # focus on ±2 SDs
# zscore_scaled = (zscore_clipped - zscore_clipped.min()) / (zscore_clipped.max() - zscore_clipped.min())


# # Define a custom red-gray-blue colormap
# colors = [
#     (0.0, "#d62728"),  # red (high assignment)
#     (0.5, "#7f7f7f"),  # gray (mid)
#     (1.0, "#1f77b4"),  # blue (low assignment) 
# ]
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list("red_gray_blue", colors)

# nbhd_meta_TEST['color'] = [matplotlib.colors.to_hex(cmap(v)) for v in zscore_scaled]
# # df_test['area'] = df_test['geometry'].area

# nbhd_meta_TEST.head()

In [ ]:
## alphashape

In [ ]:
# gdf_nbhd_LF = gdf_nbhd_LF.sjoin(largest_gdf, how="inner", predicate="within")
# gdf_nbhd_LF.shape

In [ ]:
# gdf_nbhd_LF = gdf_nbhd_LF[gdf_nbhd_LF['total_trx'] >= 200]
# gdf_nbhd_LF.shape

In [ ]:
# # # Normalize the values between 0 and 1
# # values = gdf_nbhd_LF['assigned_trx_pct']
# # norm = (values - values.min()) / (values.max() - values.min())

# # # Define a custom red-gray-blue colormap
# # colors = [
# #     (0.0, "#d62728"),  # red
# #     (0.5, "#7f7f7f"),  # gray
# #     (1.0, "#1f77b4"),  # blue
# # ]
# # cmap = matplotlib.colors.LinearSegmentedColormap.from_list("red_gray_blue", colors)

# # # # Apply the colormap
# # gdf_nbhd_LF['color'] = [matplotlib.colors.to_hex(cmap(v)) for v in norm]

# # # Compute area if needed
# # gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area

# # gdf_nbhd_LF.head()

# # Z-score standardization
# values = gdf_nbhd_LF['assigned_trx_pct']
# zscore = (values - values.mean()) / values.std()

# # Optionally, rescale z-scores to 0–1 for color mapping
# # This keeps color mapping compatible with the colormap range [0, 1]
# # zscore_scaled = (zscore - zscore.min()) / (zscore.max() - zscore.min())

# zscore_clipped = np.clip(zscore, -1, 1)  # focus on ±2 SDs
# zscore_scaled = (zscore_clipped - zscore_clipped.min()) / (zscore_clipped.max() - zscore_clipped.min())


# # Define a custom red-gray-blue colormap
# colors = [
#     (0.0, "#1f77b4"),  # blue (low)
#     (0.5, "#7f7f7f"),  # gray (mid)
#     (1.0, "#d62728"),  # red (high) 
# ]
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list("red_gray_blue", colors)

# gdf_nbhd_LF['color'] = [matplotlib.colors.to_hex(cmap(v)) for v in zscore_scaled]
# gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area

# gdf_nbhd_LF.head()

In [ ]:
# # save for future reference
# gdf_nbhd_LF.to_parquet(f"{path_landscape_files}/gdf_nbhd_meta.parquet")

### Hextile Niche

In [ ]:
gdf_nbhd_LF = gdf_nbhd.copy()
gdf_nbhd_LF = gdf_nbhd_LF[['geometry','name','leiden']]
gdf_nbhd_LF.rename(columns={'leiden':'cat'}, inplace=True)
gdf_nbhd_LF.head()

In [ ]:
categories = gdf_nbhd_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_nbhd_LF['color'] = gdf_nbhd_LF['cat'].astype(str).map(cat_to_hex)
gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area

In [ ]:
# # save for future reference

# gdf_nbhd_LF.to_parquet(f"{path_landscape_files}/gdf_nbhd.parquet")
# gdf_nbhd_T.to_parquet(f"{path_landscape_files}/nbp.parquet")
# meta_col.to_parquet(f"{path_landscape_files}/meta_col.parquet")
# meta_row.to_parquet(f"{path_landscape_files}/meta_row.parquet")

## Trx-assignment per Hextile Landscape View 

In [ ]:
landscape_ist = dega.viz.Landscape(
    technology = technology,
    base_url = base_url,
    nbhd = df_test
)

In [ ]:
landscape_ist

## Cell-Type By Hextile-Niche Landscape-Clustergram View 

In [ ]:
meta_cluster = pd.read_parquet(path_landscape_files + '/cell_clusters/meta_cluster.parquet')

In [ ]:
gdf_alpha = dega.nbhd.alpha_shape_cell_clusters(adata=adata, cat='leiden', alphas=[100, 150, 200, 250, 300, 350], meta_cluster=meta_cluster)
gdf_alpha_viz = gdf_alpha[gdf_alpha['inv_alpha'] == 100]
gdf_alpha_viz.head()

In [ ]:
# geojson_alpha = dega.nbhd.alpha_shapes.alpha_shape_geojson(gdf_alpha, meta_cluster, inst_alpha=250)

In [ ]:
landscape_ist = dega.viz.Landscape(
    technology = technology,
    base_url = base_url,
    nbhd = gdf_alpha_viz
)
dega.viz.landscape_clustergram(landscape_ist, cgm)
# landscape_ist

In [ ]:
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"
landscape_ist = dega.viz.Landscape(
    technology = "Xenium",
    base_url = base_url,
    nbhd = gdf_nbhd_LF
)
# dega.viz.landscape_clustergram(landscape_ist, cgm)
landscape_ist

## SpaGCN

In [ ]:
# !pip install --upgrade anndata scanpy h5py

In [ ]:
import anndata as ad

spagcn_annotations = ad.read_h5ad("data/spagcn_full_data_results.h5ad")
spagcn_annotations

In [ ]:
spagcn_annotations.obs['x'] = spagcn_annotations.obsm['x_pixel']
spagcn_annotations.obs['y'] = spagcn_annotations.obsm['y_pixel']

In [ ]:
from shapely.geometry import Polygon, MultiPolygon

spagcn_polygons = []

for c, group in spagcn_annotations.obs.groupby('refined_pred'):
    coords = group[['x', 'y']].to_numpy()

    # Skip clusters with too few points
    if len(coords) < 4:
        print(f"Skipping cluster {c}: only {len(coords)} points.")
        spagcn_polygons.append(None)
        continue

    # Compute alpha shape
    result = dega.nbhd.alpha_shape(coords, 50)
    print(f"Cluster {c} result type: {type(result)}")

    # Handle MultiPolygon results safely
    if isinstance(result, MultiPolygon):
        if len(result.geoms) > 0:
            result = max(result.geoms, key=lambda p: p.area)
        else:
            print(f"Cluster {c} produced an empty MultiPolygon — skipping.")
            result = None

    # Keep Polygon results as-is
    elif isinstance(result, Polygon):
        pass

    # Handle any other unexpected return types
    else:
        print(f"Cluster {c} returned non-polygon geometry ({type(result)}).")
        result = None

    spagcn_polygons.append(result)

In [ ]:
cluster_names = []
for c, group in spagcn_annotations.obs.groupby('refined_pred'):
    cluster_names.append(c)

In [ ]:
# Build the GeoDataFrame
gdf_spagcn = gpd.GeoDataFrame({
    'cat': cluster_names,
    'name': cluster_names,
    'geometry': spagcn_polygons
})
gdf_spagcn.head()

In [ ]:
import matplotlib

gdf_spagcn['cat'] = gdf_spagcn['cat'].astype('category')

categories = gdf_spagcn['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))
gdf_spagcn['color'] = gdf_spagcn['cat'].map(cat_to_hex)
gdf_spagcn['area'] = gdf_spagcn['geometry'].area
gdf_spagcn.head()

In [ ]:
gdf_spagcn = gdf_spagcn.dropna(subset=['geometry'])

In [ ]:
gdf_spagcn

In [ ]:
path_landscape_files = "data/landscape_files/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs"

In [ ]:
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"
landscape_ist = dega.viz.Landscape(
    technology = "Xenium",
    base_url = base_url,
    nbhd = gdf_spagcn
)
# dega.viz.landscape_clustergram(landscape_ist, cgm)
landscape_ist

## transfering trx annotations to cells, points2tregions

In [ ]:
# Load h5ad file, if exists already
trx = pd.read_parquet("../../points2regions/data/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs/transcripts.parquet")
trx.set_index("transcript_id", inplace=True)

In [ ]:
trx.head()

In [ ]:
p2r_annotations = pd.read_parquet("data/mouse_brain_xen_exp_P2R_120.parquet")
p2r_annotations.set_index("transcript_id", inplace=True)

In [ ]:
p2r_annotations['cell_id'] = p2r_annotations.index.map(trx['cell_id'])

In [ ]:
p2r_annotations.head()

In [ ]:
-1 in trx['cell_id'].tolist()

In [ ]:
dominant_map = (
    p2r_annotations.groupby('cell_id')['Clusters']
    .agg(lambda x: x.value_counts().idxmax())
)

In [ ]:
dominant_map

In [ ]:
# Load h5ad file, if exists already
adata = sc.read_h5ad("data/adata.h5ad")

In [ ]:
adata.obs['leiden'] = adata.obs['cell_id'].map(dominant_map).fillna(-1).astype(int)

In [ ]:
adata.obs

In [ ]:
# get cells, check the trx assignment in them and assign them as clusters?

In [ ]:
p2r_polygons = []

for c, group in p2r_annotations.groupby('Clusters'):
    coords = group[['x_location', 'y_location']].to_numpy()

    result = dega.nbhd.alpha_shape(coords, 100)
    
    if isinstance(result, MultiPolygon):
        result = max(result.geoms, key=lambda p: p.area)
    elif isinstance(result, Polygon):
        pass 
    else:
        result = None

    p2r_polygons.append(result)

In [ ]:
cluster_names = []
for c, group in p2r_annotations.groupby('Clusters'):
    cluster_names.append(c)

In [ ]:
# Build the GeoDataFrame
gdf_p2r = gpd.GeoDataFrame({
    'cat': cluster_names,
    'name': cluster_names,
    'geometry': p2r_polygons
})
gdf_p2r.head()

## p2r geojson

In [ ]:
p2r_polygons_2 = gpd.read_parquet("data/p2r_multipolygons.parquet")

In [ ]:
p2r_polygons_2['cat'] = p2r_polygons_2.index
p2r_polygons_2['name'] = p2r_polygons_2.index
p2r_polygons_2.drop(["classification", "color", "isLocked"], axis=1, inplace=True)

In [ ]:
p2r_polygons_2.rename(columns={'color_hex': 'color'}, inplace=True)
p2r_polygons_2.head()

In [ ]:
p2r_polygons_2['geometry'][0].area

In [ ]:
import matplotlib.pyplot as plt

row = p2r_polygons_2.iloc[0]
gpd.GeoDataFrame([row]).plot(color="black", facecolor="none")
plt.show()

In [ ]:
largest = max(row['geometry'].geoms, key=lambda p: p.area)

In [ ]:
largest

In [ ]:
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon

def outermost_boundary(geom):
    if geom.is_empty:
        return geom

    # For MultiPolygon, dissolve all parts into one unified shape
    merged = unary_union(geom) if geom.geom_type == "MultiPolygon" else geom

    # Take only the exterior of the merged polygon
    if merged.geom_type == "Polygon":
        return Polygon(merged.exterior)
    elif merged.geom_type == "MultiPolygon":
        # Get the largest exterior (outermost)
        largest = max(merged.geoms, key=lambda p: p.area)
        return Polygon(largest.exterior)
    return geom

# Apply across the column
p2r_polygons_2["geometry"] = p2r_polygons_2["geometry"].apply(outermost_boundary)


In [ ]:
# def remove_holes(geom):
#     if geom.is_empty:
#         return geom
#     if geom.geom_type == "Polygon":
#         return Polygon(geom.exterior)
#     if geom.geom_type == "MultiPolygon":
#         return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])
#     return geom

# p2r_polygons_2["geometry"] = p2r_polygons_2["geometry"].apply(remove_holes)

In [ ]:
print(p2r_polygons_2.is_valid.all())     
print(p2r_polygons_2.geometry.iloc[0].area)  

In [ ]:
p2r_polygons_2.plot()

In [ ]:
p2r_polygons_2['area'] = p2r_polygons_2["geometry"].area
p2r_polygons_2.head()

In [ ]:
path_landscape_files = "data/landscape_files/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs"

In [ ]:
path_landscape_files

In [ ]:
# import matplotlib

# gdf_p2r['cat'] = gdf_p2r['cat'].astype('category')

# categories = gdf_p2r['cat'].cat.categories
# n_cats = len(categories)

# cmap = matplotlib.colormaps['tab20']
# colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

# cat_to_hex = dict(zip(categories, colors))
# gdf_p2r['color'] = gdf_p2r['cat'].map(cat_to_hex)
# gdf_p2r['area'] = gdf_p2r['geometry'].area
# gdf_p2r.head()

In [ ]:
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"
landscape_ist = dega.viz.Landscape(
    technology = "Xenium",
    base_url = base_url
)
# dega.viz.landscape_clustergram(landscape_ist, cgm)
landscape_ist

In [ ]:
# base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"
# landscape_ist = dega.viz.Landscape(
#     technology = "Xenium",
#     base_url = base_url,
#     nbhd = gdf_p2r
# )
# dega.viz.landscape_clustergram(landscape_ist, cgm)
# # landscape_ist

## GASTON

In [ ]:
# Load h5ad file, if exists already
adata = sc.read_h5ad("data/adata.h5ad")

In [ ]:
adata.obs.rename(columns={"gaston_labels":"leiden"}, inplace=True)

In [ ]:
adata.obs['leiden'] = adata.obs['leiden'].astype(int)

In [ ]:
adata.obs

In [ ]:
## GASTON polygons

# gaston_annnotations = pd.read_parquet("data/gaston_annotations.parquet")

# gaston_polygons = []

# for c, group in gaston_annnotations.groupby('gaston_labels'):
#     coords = group[['x_centroid', 'y_centroid']].to_numpy()

#     result = dega.nbhd.alpha_shape(coords, 100)
    
#     if isinstance(result, MultiPolygon):
#         result = max(result.geoms, key=lambda p: p.area)
#     elif isinstance(result, Polygon):
#         pass 
#     else:
#         result = None

#     gaston_polygons.append(result)

# cluster_names = []
# for c, group in gaston_annnotations.groupby('gaston_labels'):
#     cluster_names.append(c)

# # Build the GeoDataFrame
# gdf_gaston = gpd.GeoDataFrame({
#     'cat': cluster_names,
#     'name': cluster_names,
#     'geometry': gaston_polygons
# })
# gdf_gaston.head()

# import matplotlib

# gdf_gaston['cat'] = gdf_gaston['cat'].astype('category')

# categories = gdf_gaston['cat'].cat.categories
# n_cats = len(categories)

# cmap = matplotlib.colormaps['tab20']
# colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

# cat_to_hex = dict(zip(categories, colors))
# gdf_gaston['color'] = gdf_gaston['cat'].map(cat_to_hex)
# gdf_gaston['area'] = gdf_gaston['geometry'].area
# gdf_gaston.head()

In [ ]:
adata = sc.read_h5ad("data/graphst_leiden_domains.h5ad")

In [ ]:
adata.obs

In [ ]:
path_landscape_files = "data/landscape_files/Xenium_V1_FF_Mouse_Brain_Coronal_Subset_CTX_HP_outs"
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"
landscape_ist = dega.viz.Landscape(
    technology = "Xenium",
    base_url = base_url,
    adata = adata
)
# dega.viz.landscape_clustergram(landscape_ist, cgm)
landscape_ist